# JP-1 C0 診断：日本語の例文の区切り方を見る（CPU だけ・ベンチマークは読みません）

C0 で、計画書の付録C の日本語の例文2つ（J3「割る」、J13「万年筆」）が、見込みと違う区切り方になりました。どう区切られたか（語・品詞・見出し語）を記録して、Claude が直し方を決めます。MGSM はファイルの名前の一覧だけを見ます。

**使い方**：ランタイムは **CPU**。シークレットは `GITHUB_TOKEN`。「すべてのセルを実行」で 3 分くらい。結果は Drive の `DECISION_POINTS/jp1/C0/JP1_C0_diag.json`。

- **通知**：スマホの ntfy アプリで `godpork-b06080cfb3cdfa8f` を購読しておくと、始まったとき・終わったとき・エラーで止まったときに届きます。ランタイムごと切れたときは届かないので、そのときは Colab を見てください。
- **終わったら自動でランタイムを切ります**（CU の節約）。Drive への保存を確定してから切るので、結果は残ります。エラーで止まったときは切らずに残します。切りたくないときは、最初に `AUTO_DISCONNECT = False` を実行してください。
- **エラーのとき**：通知の本文にエラーの文面（トレースバック）が入ります。全文は Drive の `DECISION_POINTS/_errors/` にも保存されるので、Claude が直接読めます。

In [ ]:
# Cell 1 — Drive, exact head, notifications.
from google.colab import drive, userdata
from pathlib import Path
import os, shutil, stat, subprocess, sys, json, time, hashlib, datetime as dt

# 通知（ntfy）：スマホの ntfy アプリでこの名前を購読しておくと、始まったとき・終わったとき・止まったときに届きます。エラーのときはエラーの文面も送ります（鍵・トークンらしい文字は伏せ字。全文は Drive の DECISION_POINTS/_errors）
import urllib.request
NTFY_TOPIC = 'godpork-b06080cfb3cdfa8f'
NB_LABEL = 'JP-1 C0 診断（形態素の確かめ・CPU）'   # どのノートブックからの通知かの名札
def notify(msg, tags='pig'):
    try:
        req = urllib.request.Request('https://ntfy.sh/' + NTFY_TOPIC, data=('[' + NB_LABEL + '] ' + msg).encode('utf-8'),
                                     headers={'Title': 'Pork - JP-1 C0 diag', 'Tags': tags})
        urllib.request.urlopen(req, timeout=10).read()
        print('通知を送りました：', msg)
    except Exception as e:
        print('通知は送れませんでした（実験には影響しません）：', type(e).__name__)
import os, re as _re, traceback as _tb, datetime as _edt
ERR_SLUG = 'jp1_c0_diag'   # Drive のエラーファイルの名前の頭
_SECRET_RE = _re.compile(r"sk-ant-[A-Za-z0-9_\-]+|github_pat_[A-Za-z0-9_]+|gh[pousr]_[A-Za-z0-9]{8,}|hf_[A-Za-z0-9]{8,}|x-access-token:[^@\s]+@")
def _scrub(s):
    s = _SECRET_RE.sub('[伏せ字]', s)
    return '\n'.join(l if len(l) <= 300 else l[:300] + ' …（長いので省略）' for l in s.splitlines())
def _error_text(result):
    e = result.error_in_exec if result.error_in_exec is not None else result.error_before_exec
    tb = ''.join(_tb.format_exception(type(e), e, e.__traceback__))
    first = (getattr(getattr(result, 'info', None), 'raw_cell', '') or '').strip().splitlines()[:1]
    return _scrub('セル: ' + (first[0][:100] if first else '?') + '\n' + tb)
def _save_error(text):
    base = '/content/drive/MyDrive'
    if not os.path.isdir(base):
        return None
    d = os.path.join(base, 'DECISION_POINTS', '_errors')
    os.makedirs(d, exist_ok=True)
    stem = ERR_SLUG + '_' + _edt.datetime.now(_edt.timezone(_edt.timedelta(hours=9))).strftime('%Y%m%d_%H%M%S')
    name, n = stem + '.txt', 1
    while os.path.exists(os.path.join(d, name)):
        n += 1
        name = stem + '_' + str(n) + '.txt'
    with open(os.path.join(d, name), 'w', encoding='utf-8') as fh:
        fh.write('[' + NB_LABEL + ']\n' + text)
    return 'DECISION_POINTS/_errors/' + name
def _short(text, limit=3300):
    b = text.encode('utf-8')
    if len(b) <= limit:
        return text
    return '…（前のほうは省略。全文は Drive）\n' + b[-limit:].decode('utf-8', 'ignore')
def _on_error(result):
    if result.error_in_exec is None and result.error_before_exec is None:
        return
    try:
        text = _error_text(result)
    except Exception as e2:
        text = 'エラーの文面を作れませんでした：' + type(e2).__name__
    try:
        where = _save_error(text)
    except Exception as e3:
        where = None
        text += '\n（Drive に保存できませんでした：' + type(e3).__name__ + '）'
    print('エラーの全文を通知に送ります' + ('（Drive：' + where + '）' if where else ''))
    notify('止まりました：エラーが出ています。' + ('全文は Drive の ' + where if where else 'Drive には保存できませんでした')
           + '\n\n' + _short(text), tags='warning')
if not globals().get('_HOOK_ON'):
    get_ipython().events.register('post_run_cell', _on_error); _HOOK_ON = True
notify('スタートしました（これが届けば通知の設定は OK です）', tags='rocket')

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

drive.mount("/content/drive")
EXPECTED_HEAD = "de53fa7b8cfdf01aea3a7c96b0365b9162658e00"
REPO_URL = "https://github.com/chichichi2001/Pork.git"
REPO = Path("/content/Pork")

def get_secret(name):
    try:
        v = userdata.get(name)
    except Exception:
        v = None
    return str(v).strip() if v and str(v).strip() else None

token = get_secret("GITHUB_TOKEN")
assert token, "Colab Secrets に GITHUB_TOKEN を登録してください。"
askpass = Path("/tmp/pork_git_askpass.sh")
askpass.write_text('#!/bin/sh\ncase "$1" in\n  *Username*) echo "x-access-token" ;;\n  *Password*) echo "$GITHUB_TOKEN" ;;\n  *) echo "" ;;\nesac\n')
askpass.chmod(askpass.stat().st_mode | stat.S_IXUSR)
env = dict(os.environ, GITHUB_TOKEN=token, GIT_ASKPASS=str(askpass), GIT_TERMINAL_PROMPT="0")
os.chdir('/content')   # セル1 をやり直しても、消すフォルダの中にいないように（2026-10-14）
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "--branch", "decision-points", "--single-branch", REPO_URL, str(REPO)], check=True, env=env)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", EXPECTED_HEAD], check=True)
head = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
assert head == EXPECTED_HEAD, (head, EXPECTED_HEAD)
token = None
env.pop("GITHUB_TOKEN", None)
os.chdir(REPO)
sys.path.insert(0, str(REPO))
D = Path("/content/drive/MyDrive/DECISION_POINTS/jp1/C0")
D.mkdir(parents=True, exist_ok=True)
JST = dt.timezone(dt.timedelta(hours=9))
if shutil.which("nvidia-smi") and subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0:
    print("⚠️ GPU ランタイムになっています。C0 は CPU だけで足ります（CU の節約のため、CPU に切り替えてね）。")

In [ ]:
# Cell 2 — how fugashi + unidic-lite splits the plan's appendix C sentences, and the MGSM file names.
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "fugashi", "unidic-lite"], capture_output=True, text=True)
assert r.returncode == 0, r.stderr[-3000:]
import importlib.metadata as md
import fugashi
from pork import jp1_text as J
TAGGER = fugashi.Tagger()
ROWS = {}
for cid, lang, sent, specs in J.APPENDIX_C:
    if lang != "ja":
        continue
    ROWS[cid] = [{"surface": w.surface, "pos1": J._feature(w, "pos1"), "pos2": J._feature(w, "pos2"),
                  "pos3": J._feature(w, "pos3"), "lemma": J._feature(w, "lemma"),
                  "category": J.ja_category(w.surface, J._feature(w, "pos1"), J._feature(w, "pos2"), J._feature(w, "lemma"))}
                 for w in TAGGER(sent)]
for cid in ("J3", "J13"):
    print(cid, [(x["surface"], x["pos1"], x["pos2"], x["lemma"], x["category"]) for x in ROWS[cid]])
from huggingface_hub import HfApi
dinfo = HfApi().dataset_info("juletxara/mgsm", token=False)
MGSM_FILES = sorted(s.rfilename for s in (dinfo.siblings or []))
print("MGSM files:", MGSM_FILES)
RESULT = {"schema": "jp1_c0_diag_v1", "head": EXPECTED_HEAD, "finished_jst": dt.datetime.now(JST).isoformat(timespec="seconds"),
          "versions": {p: md.version(p) for p in ("fugashi", "unidic-lite", "huggingface_hub")},
          "mgsm_sha": dinfo.sha, "mgsm_files": MGSM_FILES, "ja_morphemes": ROWS}
path = D / "JP1_C0_diag.json"
path.write_text(json.dumps(RESULT, ensure_ascii=False, indent=1, sort_keys=True), encoding="utf-8")
print("保存しました：", path, f"({path.stat().st_size} bytes)")
notify("C0 診断 終わりました", tags='tada')
if globals().get('AUTO_DISCONNECT', True):
    print('Drive への書き込みを確定してから、ランタイムを切ります（CU の節約。止めたいときは AUTO_DISCONNECT = False）')
    from google.colab import drive as _drive, runtime as _runtime
    _drive.flush_and_unmount()
    time.sleep(3)
    _runtime.unassign()